## 라이브러리 불러오기

In [167]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

## 데이터 불러오기


In [168]:
data_file_path = 'd:\\proj_mlops\\ajou_mlops_hw02\\data\\raw\\'
data_file = 'titanic.csv'
full_path = data_file_path + data_file
# print(full_path)
df = pd.read_csv(full_path)
display(df)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
...,...,...,...,...,...,...,...,...,...,...,...,...
886,887,0,2,"Montvila, Rev. Juozas",male,27.0,0,0,211536,13.0000,NaN,S
887,888,1,1,"Graham, Miss. Margaret Edith",female,19.0,0,0,112053,30.0000,B42,S
888,889,0,3,"Johnston, Miss. Catherine Helen ""Carrie""",female,NaN,1,2,W./C. 6607,23.4500,NaN,S
889,890,1,1,"Behr, Mr. Karl Howell",male,26.0,0,0,111369,30.0000,C148,C


In [169]:
df.isnull().sum()
# 결측치: age 177, cabin 687, Embarked 2 

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer


In [171]:
## 데이터 분리
# X, y 분리
target = 'Survived'
X = df.drop(columns=target)
y = df[target]

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(X_train.shape, y_train.shape, X_test.shape, y_test.shape)

(712, 11) (712,) (179, 11) (179,)


In [172]:
num_features = X_train.select_dtypes(include=[np.number]).columns.tolist()
cat_features = ['Pclass', 'Sex']

In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler

num_pipe_none = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

num_pipe_std = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

num_pipe_minmax = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", MinMaxScaler())
])

In [175]:
from sklearn.preprocessing import FunctionTransformer
def add_has_cabin(x):
    x = x.copy()
    x['HasCabin'] = x['Cabin'].notnull().astype(int)
    return x.drop(columns='Cabin')

cabin_pipe = Pipeline([
    ('has_cabin', FunctionTransformer(add_has_cabin))
])

In [176]:
drop_cols = ['Embarked', 'Name', 'Ticket', 'PassengerId']

def drop_columns(X, columns=drop_cols):
    return X.drop(columns=columns)

drop_cols_pipe = Pipeline([
    ('drop_cols', FunctionTransformer(drop_columns, kw_args={'columns': drop_cols}))
])

In [177]:
from sklearn.preprocessing import OneHotEncoder

# 범주형(Pclass, Sex) one-hot Encoding
cat_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

In [ ]:
common_transformers = [
    ("cat_drop", drop_cols_pipe, drop_cols),
    ("cat", cat_pipe, cat_features),
]

preprocess_none = ColumnTransformer([
    ("num", num_pipe_none, num_features),
    *common_transformers
])

preprocess_std = ColumnTransformer([
    ("num", num_pipe_std, num_features),
    *common_transformers
])

preprocess_minmax = ColumnTransformer([
    ("num", num_pipe_minmax, num_features),
    *common_transformers
])

## 모델 학습

In [143]:
# SVC 학습과 모델 평가를 위한 라이브러리
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report

### SVM without scaling

In [144]:
# 모델 생성
svm_clf_wo_scale = SVC(random_state=42)
svm_clf_wo_scale.fit(X_train_enc, y_train)

# 예측
y_pred_svm_wo_scale = svm_clf_wo_scale.predict(X_test_enc)

#평가
print('Acc: ', accuracy_score(y_test, y_pred_svm_wo_scale))
print(classification_report(y_test, y_pred_svm_wo_scale))

Acc:  0.6536312849162011
              precision    recall  f1-score   support

           0       0.64      0.94      0.76       105
           1       0.75      0.24      0.37        74

    accuracy                           0.65       179
   macro avg       0.69      0.59      0.56       179
weighted avg       0.68      0.65      0.60       179



### SVM with StandardScaler

In [ ]:
# 모델 생성
svm_clf = SVC(random_state=42)
svm_clf.fit(X_train_scale, y_train)

# 예측
y_pred_svm = svm_clf.predict(X_test_scale)

#평가
print('Acc: ', accuracy_score(y_test, y_pred_svm))
print(classification_report(y_test, y_pred_svm))

Acc:  0.8100558659217877
              precision    recall  f1-score   support

           0       0.81      0.88      0.84       105
           1       0.80      0.72      0.76        74

    accuracy                           0.81       179
   macro avg       0.81      0.80      0.80       179
weighted avg       0.81      0.81      0.81       179



### SVM with MinMaxScaler

In [148]:
# 모델 생성
svm_clf_mms = SVC(random_state=42)
svm_clf_mms.fit(X_train_mms, y_train)

# 예측
y_pred_svm_mms = svm_clf_mms.predict(X_test_mms)

#평가
print('Acc: ', accuracy_score(y_test, y_pred_svm_mms))
print(classification_report(y_test, y_pred_svm_mms))

Acc:  0.7932960893854749
              precision    recall  f1-score   support

           0       0.80      0.87      0.83       105
           1       0.78      0.69      0.73        74

    accuracy                           0.79       179
   macro avg       0.79      0.78      0.78       179
weighted avg       0.79      0.79      0.79       179



### KNN with StandardScaler

In [141]:
from sklearn.neighbors import KNeighborsClassifier

knn_clf = KNeighborsClassifier(n_neighbors=5)
knn_clf.fit(X_train_scale, y_train)

y_pred_knn = knn_clf.predict(X_test_scale)

print('KNN acc: ', accuracy_score(y_test, y_pred_knn));
print(classification_report(y_test, y_pred_knn))

KNN acc:  0.8212290502793296
              precision    recall  f1-score   support

           0       0.83      0.88      0.85       105
           1       0.81      0.74      0.77        74

    accuracy                           0.82       179
   macro avg       0.82      0.81      0.81       179
weighted avg       0.82      0.82      0.82       179



### KNN with MinMaxScaler

In [149]:
knn_clf_mms = KNeighborsClassifier(n_neighbors=5)
knn_clf_mms.fit(X_train_mms, y_train)

y_pred_knn_mms = knn_clf_mms.predict(X_test_mms)

print('KNN acc: ', accuracy_score(y_test, y_pred_knn_mms));
print(classification_report(y_test, y_pred_knn_mms))

KNN acc:  0.8044692737430168
              precision    recall  f1-score   support

           0       0.81      0.87      0.84       105
           1       0.79      0.72      0.75        74

    accuracy                           0.80       179
   macro avg       0.80      0.79      0.80       179
weighted avg       0.80      0.80      0.80       179



## 결과 비교

In [ ]:
print("Accuracy")
print('='*30)
print('SVM(No scale): ', np.round(accuracy_score(y_test, y_pred_svm_wo_scale), 3))
print('SVM(StandardScaler): ', np.round(accuracy_score(y_test, y_pred_svm), 3))
print('SVM(MinMaxScaler): ', np.round(accuracy_score(y_test, y_pred_svm_mms), 3))
print('KNN(StandardScaler): ', np.round(accuracy_score(y_test, y_pred_knn), 3))
print('KNN(MinMaxScaler): ', np.round(accuracy_score(y_test, y_pred_knn_mms), 3))
print('='*30)
print("Classification Report")
print('='*30)
print("1. SVM(No scale)")
print(classification_report(y_test, y_pred_svm_wo_scale))
print("SVM(StandardScaler)")
print(classification_report(y_test, y_pred_svm))
print("SVM(MinMaxScaler)")
print(classification_report(y_test, y_pred_svm_mms))
print("KNN(StandardScaler)")
print(classification_report(y_test, y_pred_knn))
print("KNN(MinMaxScaler)")
print(classification_report(y_test, y_pred_knn_mms))


# 사용 예시:
# X_dropped = drop_cols_pipe.fit_transform(X)

Accuracy
SVM(No scale):  0.654
SVM(StandardScaler):  0.81
SVM(MinMaxScaler):  0.793
KNN(StandardScaler):  0.821
KNN(MinMaxScaler):  0.804
Classification Report
1. SVM(No scale)
              precision    recall  f1-score   support

           0       0.64      0.94      0.76       105
           1       0.75      0.24      0.37        74

    accuracy                           0.65       179
   macro avg       0.69      0.59      0.56       179
weighted avg       0.68      0.65      0.60       179

SVM(StandardScaler)
              precision    recall  f1-score   support

           0       0.81      0.88      0.84       105
           1       0.80      0.72      0.76        74

    accuracy                           0.81       179
   macro avg       0.81      0.80      0.80       179
weighted avg       0.81      0.81      0.81       179

SVM(MinMaxScaler)
              precision    recall  f1-score   support

           0       0.80      0.87      0.83       105
           1       0.7